# IT23183186 Model A — Varroa MobileNetV2

Notebook interface for the existing, reproducible Model A pipeline. It runs the repository scripts so the notebook and command-line workflow use the same implementation. Run cells in order. Dataset inspection and preparation are safe to run; start training and test evaluation only when ready. The current checkpoint is partial and needs retraining.

## 1. Configure paths and dependencies

Install this model environment once, if needed, using the notebook kernel: `python -m pip install -r backend/models/IT23183186_model_a_varroa_classifier/requirements.txt`. Restart the kernel after installing.

Set `DATASET_SOURCE` to the folder containing the official `train.zip`, `val.zip`, `test.zip`, and `gt.csv`. Original downloads are read-only; prepared data, reports, and checkpoints are written to separate run folders. Run cells in order. Training and final test evaluation are explicit later steps.

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

MODEL_RELATIVE_DIR = Path('backend') / 'models' / 'IT23183186_model_a_varroa_classifier'

def find_repository_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / MODEL_RELATIVE_DIR).is_dir():
            return candidate
    raise FileNotFoundError(
        f'Could not find the repository above {start}. Start Jupyter inside the project folder.'
    )

REPO_ROOT = find_repository_root()
MODEL_DIR = REPO_ROOT / MODEL_RELATIVE_DIR

# Keep this source path pointed at your unchanged official download folder.
DATASET_SOURCE = Path(r'C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_original')
PREPARED_BASE_DIR = Path(r'C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_prepared')
RUN_ID = datetime.now().strftime('IT23183186_ModelA_%Y%m%d_%H%M%S')
PREPARED_DIR = PREPARED_BASE_DIR / RUN_ID
REPORT_DIR = MODEL_DIR / 'reports' / RUN_ID
ARTIFACTS_DIR = MODEL_DIR / 'artifacts' / RUN_ID
for folder in (PREPARED_DIR, REPORT_DIR, ARTIFACTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

PYTHON = sys.executable

import runpy
_notebook_support = runpy.run_path(str(MODEL_DIR / 'notebook_support.py'))

def run(args):
    return _notebook_support['run_logged'](args, cwd=REPO_ROOT, log_dir=REPORT_DIR / 'logs')

print('Repository root:', REPO_ROOT)
print('Model scripts:', MODEL_DIR)
print('Dataset exists:', DATASET_SOURCE.is_dir())
print('Prepared output:', PREPARED_DIR)
print('Run artifacts:', ARTIFACTS_DIR)


## 2. Inspect the official dataset

Reports class counts, archive contents, and source metadata. Update `DATASET_SOURCE` above to your actual location first.

In [5]:
run([PYTHON, MODEL_DIR / 'inspect_dataset.py', '--source', DATASET_SOURCE, '--verify-checksums', '--output', REPORT_DIR / 'dataset_inspection.json'])

Running: c:\Users\User\anaconda3\python.exe C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\inspect_dataset.py --source C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_original --verify-checksums --output C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\reports\IT23183186_ModelA_20261020_170719\dataset_inspection.json


## 3. Prepare and audit

The script preserves original files, writes prepared copies/manifests separately, and reports exclusions and duplicates.

In [6]:
run([PYTHON, MODEL_DIR / 'prepare_dataset.py', '--source', DATASET_SOURCE, '--output', PREPARED_DIR, '--report-copy', REPORT_DIR / 'preparation_report.json'])

Running: c:\Users\User\anaconda3\python.exe C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\prepare_dataset.py --source C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_original --output C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_prepared\IT23183186_ModelA_20261020_170719 --report-copy C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\reports\IT23183186_ModelA_20261020_170719\preparation_report.json


## 4. Train Model A

This runs the reproducible transfer-learning script: train the classification head, then fine-tune selected backbone blocks. Early stopping and best validation checkpoint saving are configured by the script. The run-specific output directory prevents overwriting earlier checkpoints. Training does not use the test split for model selection.

### Training environment preflight and visible logs
Use this preflight after sections 1-3. The training cell can also run first after a kernel restart: it restores its explicitly selected PREPARED_DIR and creates fresh output folders. Change that path to use a different prepared dataset.


In [11]:
# A restarted kernel does not retain variables from earlier cells.
_required = ('MODEL_DIR', 'REPO_ROOT', 'PREPARED_DIR', 'ARTIFACTS_DIR', 'REPORT_DIR', 'sys')
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        'Notebook setup is missing from this kernel: ' + ', '.join(_missing)
        + '. Run section 1 (Configure paths and dependencies), then sections 2 and 3 '
        '(Inspect and Prepare), then the training preflight and training cells in order.'
    )

# Run this cell before training. It preserves existing run directories and prepared data.
import runpy
_notebook_support = runpy.run_path(str(MODEL_DIR / 'notebook_support.py'))
PYTHON = _notebook_support['select_training_python']([
    MODEL_DIR / '.venv' / 'Scripts' / 'python.exe',
    sys.executable,
    REPO_ROOT / 'backend' / '.venv' / 'Scripts' / 'python.exe',
])

def run(args):
    return _notebook_support['run_logged'](args, cwd=REPO_ROOT, log_dir=REPORT_DIR / 'logs')

missing = [str(PREPARED_DIR / 'manifests' / f'{split}.csv')
           for split in ('train', 'val', 'test')
           if not (PREPARED_DIR / 'manifests' / f'{split}.csv').is_file()]
if missing:
    raise FileNotFoundError('Prepared manifests are missing: ' + ', '.join(missing))
print('Prepared manifests exist. Ready for the explicit training cell; no training has started.')


Training Python: C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\.venv\Scripts\python.exe
torch 2.7.1+cpu torchvision 0.22.1+cpu
Prepared manifests exist. Ready for the explicit training cell; no training has started.


In [ ]:
# Restore setup when this is the first cell after a kernel restart.
from datetime import datetime
from pathlib import Path
import runpy
import sys

if not callable(globals().get('run')) or any(
    name not in globals() for name in ('PYTHON', 'MODEL_DIR', 'PREPARED_DIR', 'ARTIFACTS_DIR')
):
    _relative = Path('backend/models/IT23183186_model_a_varroa_classifier')
    _cwd = Path.cwd().resolve()
    _roots = [root for parent in (_cwd, *_cwd.parents)
              for root in (parent, parent / 'J26-IT-378-Research-Project')
              if (root / _relative / 'notebook_support.py').is_file()]
    if not _roots:
        raise FileNotFoundError('Open this notebook inside PROJECT_V1 or the research repository.')
    REPO_ROOT = _roots[0]
    MODEL_DIR = REPO_ROOT / _relative
    # Existing prepared run. Change this path to use a different prepared dataset.
    PREPARED_DIR = Path(r'C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_prepared\IT23183186_ModelA_20261020_170719')
    _missing = [str(PREPARED_DIR / 'manifests' / f'{split}.csv')
                for split in ('train', 'val', 'test')
                if not (PREPARED_DIR / 'manifests' / f'{split}.csv').is_file()]
    if _missing:
        raise FileNotFoundError('Set PREPARED_DIR to a prepared dataset or run sections 1-3 first. Missing: ' + ', '.join(_missing))
    # Fresh output folders protect earlier checkpoints.
    RUN_ID = datetime.now().strftime('IT23183186_ModelA_%Y%m%d_%H%M%S_%f')
    REPORT_DIR = MODEL_DIR / 'reports' / RUN_ID
    ARTIFACTS_DIR = MODEL_DIR / 'artifacts' / RUN_ID
    _notebook_support = runpy.run_path(str(MODEL_DIR / 'notebook_support.py'))
    PYTHON = _notebook_support['select_training_python']([
        MODEL_DIR / '.venv' / 'Scripts' / 'python.exe',
        sys.executable,
        REPO_ROOT / 'backend' / '.venv' / 'Scripts' / 'python.exe',
    ])

    def run(args):
        return _notebook_support['run_logged'](args, cwd=REPO_ROOT, log_dir=REPORT_DIR / 'logs')

    print('Restored prepared dataset:', PREPARED_DIR)
    print('New training outputs:', ARTIFACTS_DIR)

# Run this cell when inspection and preparation have completed successfully.
run([
    PYTHON, MODEL_DIR / 'train.py',
    '--prepared-dir', PREPARED_DIR,
    '--output-dir', ARTIFACTS_DIR,
    '--seed', '2026',
    '--batch-size', '32',
    '--head-epochs', '5',
    '--finetune-epochs', '8',
    '--patience', '3',
])

Training Python: C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\.venv\Scripts\python.exe
torch 2.7.1+cpu torchvision 0.22.1+cpu
Restored prepared dataset: C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_prepared\IT23183186_ModelA_20261020_170719
New training outputs: C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\artifacts\IT23183186_ModelA_20261020_222159_360522
Running: "C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\.venv\Scripts\python.exe" "C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\PROJECT_V1\J26-IT-378-Research-Project\backend\models\IT23183186_model_a_varroa_classifier\train.py" --prepared-dir "C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDatas

## 5. Evaluate the held-out test split

Run only after training finishes successfully. This evaluates the saved checkpoint on the official test manifest, reports accuracy, per-class precision/recall/F1, confusion matrix and ROC-AUC when defined, and writes predictions. Do not use test results to tune the model.

In [ ]:
# Protect the test set while training is unfinished.
completion_files = ['training_config.json', 'training_history.csv', 'class_map.json', 'preprocessing.json']
missing = [name for name in completion_files if not (ARTIFACTS_DIR / name).is_file()]
if missing:
    raise RuntimeError('Training completion artifacts are missing. Wait for training to finish before final test evaluation: ' + ', '.join(missing))
checkpoint = ARTIFACTS_DIR / 'IT23183186_ModelA_MobileNetV2.pt'
if not checkpoint.is_file():
    raise FileNotFoundError(f'Trained checkpoint not found: {checkpoint}. Complete the training cell first.')
run([
    PYTHON, MODEL_DIR / 'evaluate.py',
    '--prepared-dir', PREPARED_DIR,
    '--model-dir', ARTIFACTS_DIR,
    '--output-dir', ARTIFACTS_DIR / 'evaluation',
])

## Display metrics and graphs
Run this cell during training for validation progress, or after the test-evaluation cell for final confusion matrix, per-class performance and ROC graphs. Requires matplotlib and numpy in the notebook kernel. It only plots saved outputs; it never starts training or test inference.


In [ ]:
import runpy
from IPython.display import Image, display
plotter = runpy.run_path(str(MODEL_DIR / 'plot_results.py'))
logs = sorted((REPORT_DIR / 'logs').glob('train_*.log'))
graph_dir = plotter['plot_results'](ARTIFACTS_DIR, logs[-1] if logs else None)
for graph in sorted(graph_dir.glob('*.png')):
    display(Image(filename=str(graph)))
